<a href="https://colab.research.google.com/github/guthasushmasree/ML-LabPrograms/blob/main/ml(skill_20).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# UNIVERSAL RANDOM FOREST HYPERPARAMETER TUNING
# ============================================================
# Works with any CSV classification dataset
#
# Includes:
# 1. CSV upload
# 2. Automatic target detection
# 3. Manual target selection if needed
# 4. Data cleaning
# 5. Grid Search
# 6. Randomized Search
# 7. Validation Curve
# 8. Learning Curve
# 9. Train-validation gap
# 10. Overfitting diagnosis
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from scipy.stats import randint, uniform

from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    validation_curve,
    learning_curve,
    StratifiedKFold
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline


# ============================================================
# 2. UPLOAD CSV FILE
# ============================================================

from google.colab import files

print("=" * 70)
print("UPLOAD YOUR CSV DATASET")
print("=" * 70)

uploaded = files.upload()

filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("\nDataset loaded successfully!")
print("File:", filename)
print("Original shape:", df.shape)


# ============================================================
# 3. DISPLAY DATASET
# ============================================================

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


# ============================================================
# 4. BASIC CLEANING
# ============================================================

# Remove completely empty columns
df = df.dropna(axis=1, how="all")

# Remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

print("\nShape after cleaning:", df.shape)


# ============================================================
# 5. DETECT TARGET COLUMN
# ============================================================

possible_targets = [
    "target",
    "label",
    "class",
    "y",
    "placed",
    "placement",
    "fraud",
    "default",
    "churn",
    "outcome",
    "result",
    "status",
    "prediction"
]

target = None


# Search for common target names
for col in df.columns:

    if col.lower().strip() in possible_targets:

        target = col
        break


# ============================================================
# 6. ASK USER IF TARGET WAS NOT FOUND
# ============================================================

if target is None:

    print("\n" + "=" * 70)
    print("TARGET COLUMN NOT DETECTED AUTOMATICALLY")
    print("=" * 70)

    print("\nAvailable columns:")

    for i, col in enumerate(df.columns):

        print(f"{i}: {col}")

    while True:

        try:

            target_index = int(
                input(
                    "\nEnter the NUMBER of the target column: "
                )
            )

            if 0 <= target_index < len(df.columns):

                target = df.columns[target_index]

                break

            else:

                print(
                    "Invalid number. Please select a valid column."
                )

        except ValueError:

            print(
                "Please enter a number."
            )


print("\nSelected target column:", target)


# ============================================================
# 7. REMOVE ROWS WITH MISSING TARGET
# ============================================================

df = df.dropna(
    subset=[target]
).reset_index(drop=True)

y = df[target]


print("\n" + "=" * 70)
print("TARGET INFORMATION")
print("=" * 70)

print("\nTarget column:", target)

print(
    "Number of unique classes:",
    y.nunique()
)

print("\nTarget distribution:")

print(
    y.value_counts()
)


# ============================================================
# 8. CHECK TARGET
# ============================================================

if y.nunique() < 2:

    raise ValueError(
        "The target column must contain at least 2 classes."
    )


# Random Forest supports multiclass classification,
# so we DO NOT reject targets with more than 20 classes.

if y.nunique() > 20:

    print(
        "\nWARNING:"
    )

    print(
        f"The target contains {y.nunique()} classes."
    )

    print(
        "Random Forest can still perform multiclass classification."
    )


# ============================================================
# 9. SEPARATE FEATURES AND TARGET
# ============================================================

X_all = df.drop(
    columns=[target]
)


# ============================================================
# 10. SELECT NUMERIC FEATURES
# ============================================================

numeric_features = X_all.select_dtypes(
    include=np.number
).columns.tolist()


print("\n" + "=" * 70)
print("NUMERIC FEATURES")
print("=" * 70)

print("\nDetected numeric features:")

for feature in numeric_features:

    print(" -", feature)


# ============================================================
# 11. CHECK NUMERIC FEATURES
# ============================================================

if len(numeric_features) == 0:

    raise ValueError(
        "No numeric features were found. "
        "This experiment requires numeric features."
    )


# ============================================================
# 12. REMOVE ID-LIKE FEATURES
# ============================================================

id_keywords = [
    "id",
    "index",
    "roll",
    "registration",
    "serial",
    "number",
    "no"
]

remove_cols = []


for col in numeric_features:

    col_clean = (
        col.lower()
        .replace("_", "")
        .replace("-", "")
        .replace(" ", "")
    )

    uniqueness = (
        X_all[col].nunique()
        /
        len(X_all)
    )


    # Remove obvious ID-like columns
    if any(
        keyword in col_clean
        for keyword in id_keywords
    ):

        if uniqueness > 0.80:

            remove_cols.append(col)


    # Remove almost completely unique columns
    elif uniqueness > 0.98:

        remove_cols.append(col)


# ============================================================
# 13. REMOVE ID COLUMNS
# ============================================================

if len(remove_cols) > 0:

    print("\n" + "=" * 70)
    print("REMOVING ID-LIKE FEATURES")
    print("=" * 70)

    print(
        "\nRemoved columns:"
    )

    for col in remove_cols:

        print(
            " -", col
        )

    numeric_features = [
        col
        for col in numeric_features
        if col not in remove_cols
    ]


# ============================================================
# 14. REMOVE CONSTANT FEATURES
# ============================================================

constant_cols = [
    col
    for col in numeric_features
    if X_all[col].nunique(dropna=True) <= 1
]


if len(constant_cols) > 0:

    print("\n" + "=" * 70)
    print("REMOVING CONSTANT FEATURES")
    print("=" * 70)

    for col in constant_cols:

        print(
            " -", col
        )

    numeric_features = [
        col
        for col in numeric_features
        if col not in constant_cols
    ]


# ============================================================
# 15. FINAL FEATURES
# ============================================================

if len(numeric_features) == 0:

    raise ValueError(
        "No useful numeric features remain after cleaning."
    )


X = X_all[
    numeric_features
].copy()


# ============================================================
# 16. CONVERT TARGET TO NUMERIC
# ============================================================

if not pd.api.types.is_numeric_dtype(y):

    print("\nTarget is categorical.")

    class_names = sorted(
        y.astype(str).unique()
    )

    mapping = {
        name: i
        for i, name in enumerate(class_names)
    }

    print("\nTarget encoding:")

    for name, number in mapping.items():

        print(
            f"{name} -> {number}"
        )

    y = (
        y.astype(str)
        .map(mapping)
    )


# ============================================================
# 17. FINAL DATASET INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET")
print("=" * 70)

print(
    "\nDataset:",
    filename
)

print(
    "Samples:",
    len(X)
)

print(
    "Features:",
    len(numeric_features)
)

print(
    "Target:",
    target
)

print("\nFeatures used:")

for feature in numeric_features:

    print(
        " -", feature
    )


# ============================================================
# 18. STRATIFIED CROSS VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=0
)


# ============================================================
# 19. RANDOM FOREST PIPELINE
# ============================================================

pipeline = Pipeline([

    (
        "imputer",

        SimpleImputer(
            strategy="median"
        )
    ),

    (
        "model",

        RandomForestClassifier(
            random_state=0
        )
    )
])


# ============================================================
# 20. GRID SEARCH
# ============================================================

print("\n" + "=" * 70)
print("GRID SEARCH")
print("=" * 70)


grid = {

    "model__n_estimators": [
        100,
        200,
        300
    ],

    "model__max_depth": [
        4,
        6,
        8,
        None
    ],

    "model__min_samples_leaf": [
        1,
        3,
        5
    ]
}


# 3 × 4 × 3 = 36 combinations

t0 = time.time()


gs = GridSearchCV(

    pipeline,

    grid,

    cv=cv,

    n_jobs=-1,

    scoring="accuracy"
)


gs.fit(
    X,
    y
)


grid_time = (
    time.time()
    -
    t0
)


grid_fits = (
    len(gs.cv_results_["params"])
    *
    cv.n_splits
)


print(
    f"\nGridSearch fits={grid_fits:<4} "
    f"time={grid_time:.1f}s "
    f"best={gs.best_score_:.4f}"
)


print(
    "\nBest GridSearch parameters:"
)

print(
    gs.best_params_
)


# ============================================================
# 21. RANDOMIZED SEARCH
# ============================================================

print("\n" + "=" * 70)
print("RANDOMIZED SEARCH")
print("=" * 70)


dist = {

    "model__n_estimators":
        randint(
            50,
            400
        ),

    "model__max_depth":
        randint(
            3,
            15
        ),

    "model__min_samples_leaf":
        randint(
            1,
            10
        ),

    "model__max_features":
        uniform(
            0.2,
            0.8
        )
}


n_iter = 20


t0 = time.time()


rs = RandomizedSearchCV(

    pipeline,

    dist,

    n_iter=n_iter,

    cv=cv,

    random_state=0,

    n_jobs=-1,

    scoring="accuracy"
)


rs.fit(
    X,
    y
)


random_time = (
    time.time()
    -
    t0
)


random_fits = (
    n_iter
    *
    cv.n_splits
)


print(
    f"\nRandomSearch fits={random_fits:<4} "
    f"time={random_time:.1f}s "
    f"best={rs.best_score_:.4f}"
)


print(
    "\nBest RandomSearch parameters:"
)

print(
    rs.best_params_
)


# ============================================================
# 22. GRID VS RANDOM SEARCH
# ============================================================

print("\n" + "=" * 70)
print("GRID SEARCH VS RANDOM SEARCH")
print("=" * 70)


comparison = pd.DataFrame({

    "Method": [

        "Grid Search",

        "Randomized Search"

    ],

    "Fits": [

        grid_fits,

        random_fits

    ],

    "Time_seconds": [

        round(
            grid_time,
            2
        ),

        round(
            random_time,
            2
        )

    ],

    "Best_CV_Accuracy": [

        round(
            gs.best_score_,
            4
        ),

        round(
            rs.best_score_,
            4
        )

    ]

})


display(
    comparison
)


# ============================================================
# 23. VALIDATION CURVE
# ============================================================

print("\n" + "=" * 70)
print("VALIDATION CURVE")
print("=" * 70)


depths = [
    1,
    2,
    3,
    4,
    6,
    8,
    10,
    14,
    None
]


train_scores, validation_scores = validation_curve(

    pipeline,

    X,

    y,

    param_name="model__max_depth",

    param_range=depths,

    cv=cv,

    scoring="accuracy",

    n_jobs=-1
)


train_mean = (
    train_scores.mean(
        axis=1
    )
)


train_std = (
    train_scores.std(
        axis=1
    )
)


validation_mean = (
    validation_scores.mean(
        axis=1
    )
)


validation_std = (
    validation_scores.std(
        axis=1
    )
)


# None cannot be plotted directly
plot_depths = [

    d if d is not None else 16

    for d in depths

]


plt.figure(
    figsize=(9, 5)
)


plt.plot(

    plot_depths,

    train_mean,

    "o-",

    label="Train"

)


plt.plot(

    plot_depths,

    validation_mean,

    "s-",

    label="Validation"

)


plt.fill_between(

    plot_depths,

    validation_mean - validation_std,

    validation_mean + validation_std,

    alpha=0.2

)


plt.xlabel(
    "max_depth"
)

plt.ylabel(
    "Accuracy"
)

plt.legend()


plt.title(
    "Validation Curve: Where Does Overfitting Start?"
)


plt.grid(
    alpha=0.3
)


plt.show()


# ============================================================
# 24. LEARNING CURVE
# ============================================================

print("\n" + "=" * 70)
print("LEARNING CURVE")
print("=" * 70)


sizes, train_scores_2, validation_scores_2 = learning_curve(

    rs.best_estimator_,

    X,

    y,

    cv=cv,

    n_jobs=-1,

    train_sizes=np.linspace(
        0.1,
        1.0,
        8
    ),

    scoring="accuracy"

)


train_mean_2 = (
    train_scores_2.mean(
        axis=1
    )
)


train_std_2 = (
    train_scores_2.std(
        axis=1
    )
)


validation_mean_2 = (
    validation_scores_2.mean(
        axis=1
    )
)


validation_std_2 = (
    validation_scores_2.std(
        axis=1
    )
)


plt.figure(
    figsize=(9, 5)
)


plt.plot(

    sizes,

    train_mean_2,

    "o-",

    label="Train"

)


plt.plot(

    sizes,

    validation_mean_2,

    "s-",

    label="Validation"

)


plt.fill_between(

    sizes,

    validation_mean_2 - validation_std_2,

    validation_mean_2 + validation_std_2,

    alpha=0.2

)


plt.xlabel(
    "Training Samples"
)

plt.ylabel(
    "Accuracy"
)

plt.legend()


plt.title(
    "Learning Curve: Would More Data Help?"
)


plt.grid(
    alpha=0.3
)


plt.show()


# ============================================================
# 25. FINAL TRAIN-VALIDATION GAP
# ============================================================

gap = (

    train_mean_2[-1]

    -

    validation_mean_2[-1]

)


print("\n" + "=" * 70)
print("FINAL DIAGNOSIS")
print("=" * 70)


print(
    f"\nFinal train-validation gap = {gap:.4f}"
)


if gap > 0.05:

    print(
        "\nDiagnosis: HIGH VARIANCE - possible overfitting"
    )

    print(
        "\nPossible solutions:"
    )

    print(
        "1. Collect more training data"
    )

    print(
        "2. Reduce model complexity"
    )

    print(
        "3. Reduce max_depth"
    )

    print(
        "4. Increase min_samples_leaf"
    )

else:

    print(
        "\nDiagnosis: CONVERGED"
    )

    print(
        "Train and validation performance are relatively close."
    )

    print(
        "More data may not provide a large improvement."
    )

    print(
        "Consider improving features or trying another model."
    )


# ============================================================
# 26. FINAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)


print(
    "\nDataset               :",
    filename
)

print(
    "Samples               :",
    len(X)
)

print(
    "Features              :",
    len(numeric_features)
)

print(
    "Target                :",
    target
)

print(
    "Number of classes     :",
    y.nunique()
)

print(
    "Grid Search Accuracy  :",
    f"{gs.best_score_:.4f}"
)

print(
    "Random Search Accuracy:",
    f"{rs.best_score_:.4f}"
)

print(
    "Grid Search Time      :",
    f"{grid_time:.2f} seconds"
)

print(
    "Random Search Time    :",
    f"{random_time:.2f} seconds"
)

print(
    "Train-Validation Gap  :",
    f"{gap:.4f}"
)


print(
    "\nBest Random Forest parameters:"
)


for parameter, value in rs.best_params_.items():

    print(
        f"{parameter}: {value}"
    )


print("\n" + "=" * 70)
print("EXPERIMENT COMPLETED SUCCESSFULLY")
print("=" * 70)

UPLOAD YOUR CSV DATASET


Saving placement_predict_50k Dataset.csv to placement_predict_50k Dataset.csv

Dataset loaded successfully!
File: placement_predict_50k Dataset.csv
Original shape: (50000, 32)

DATASET INFORMATION

Columns:
['StudentID', 'Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'CGPA_Tier', 'PlacementStatus', 'IsAnomaly', 'Salary Package']

First 5 rows:


,StudentID,Gender,City,CollegeTier,Stream,Specialisation,Hostel,HistoryOfBacklogs,SGPA_Sem1,SGPA_Sem2,...,Publications,AptitudeTestScore,SoftSkillsRating,CodingTestScore,MockInterviewScore,ExtraCurricular,CGPA_Tier,PlacementStatus,IsAnomaly,Salary Package
0,1,Male,Ahmedabad,Tier2,ECE,Networking,No,No,6.02,6.54,...,0,66.7,2.2,49.4,47.8,0,Low,0,0,0.00
1,2,Female,Mumbai,Tier2,ECE,DataScience,Yes,Yes,5.84,5.12,...,0,48.2,2.4,26.7,25.8,0,Low,0,0,0.00
2,3,Male,Kolkata,Tier2,IT,DataScience,Yes,No,4.91,5.29,...,0,73.8,2.8,67.7,41.5,0,Low,1,0,3.89
3,4,Male,Jaipur,Tier1,CS,AI,No,No,7.67,8.03,...,0,69.8,2.7,66.9,48.0,0,Mid,1,0,8.37
4,5,Male,Pune,Tier2,IT,DataScience,Yes,No,8.14,8.97,...,1,73.1,2.1,71.7,61.7,1,High,1,0,18.99



Shape after cleaning: (50000, 32)

TARGET COLUMN NOT DETECTED AUTOMATICALLY

Available columns:
0: StudentID
1: Gender
2: City
3: CollegeTier
4: Stream
5: Specialisation
6: Hostel
7: HistoryOfBacklogs
8: SGPA_Sem1
9: SGPA_Sem2
10: SGPA_Sem3
11: SGPA_Sem4
12: SGPA_Sem5
13: SGPA_Sem6
14: SGPA_Sem7
15: SGPA_Sem8
16: CGPA
17: AttendancePercent
18: Internships
19: Projects
20: Workshops
21: Certifications
22: Publications
23: AptitudeTestScore
24: SoftSkillsRating
25: CodingTestScore
26: MockInterviewScore
27: ExtraCurricular
28: CGPA_Tier
29: PlacementStatus
30: IsAnomaly
31: Salary Package

Selected target column: Hostel

TARGET INFORMATION

Target column: Hostel
Number of unique classes: 2

Target distribution:
Hostel
Yes    27785
No     22215
Name: count, dtype: int64

NUMERIC FEATURES

Detected numeric features:
 - StudentID
 - SGPA_Sem1
 - SGPA_Sem2
 - SGPA_Sem3
 - SGPA_Sem4
 - SGPA_Sem5
 - SGPA_Sem6
 - SGPA_Sem7
 - SGPA_Sem8
 - CGPA
 - AttendancePercent
 - Internships
 - Projects
 -

/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



GridSearch fits=180  time=2191.7s best=0.5558

Best GridSearch parameters:
{'model__max_depth': 8, 'model__min_samples_leaf': 3, 'model__n_estimators': 100}

RANDOMIZED SEARCH
